In [0]:
%pip install openslide-python openslide-bin tifffile imagecodecs -q

In [0]:
# ── Shared utilities — run after pip install, before any other cell ──────────
import glob, os
import openslide
from PIL import Image
import matplotlib.pyplot as plt

Image.MAX_IMAGE_PIXELS = None   # safe — we read metadata or small sub-images only

SOURCE = "/Volumes/hls_pathology/osuwmc/sample"
EXT    = ".tiff"


def discover_tiff_files(source=SOURCE, ext=EXT, synthetic=True):
    """Return sorted list of TIFF paths under *source*.
    Pass synthetic=False to exclude files whose name contains '__synthetic'.
    """
    files = sorted(set(
        glob.glob(f"{source}/**/*{ext}", recursive=True) +
        glob.glob(f"{source}/*{ext}")
    ))
    if not synthetic:
        files = [f for f in files if "__synthetic" not in f]
    return files


def to_rgb(img):
    """Flatten RGBA / palette PIL images to RGB on a white background."""
    if img.mode == "RGBA":
        bg = Image.new("RGB", img.size, (255, 255, 255))
        bg.paste(img, mask=img.split()[3])
        return bg
    return img.convert("RGB")


def fit_to(img, max_w=800, max_h=600):
    """Downsample *img* to fit within (max_w, max_h), preserving aspect ratio.
    Never upscales."""
    w, h = img.size
    scale = min(max_w / w, max_h / h, 1.0)
    if scale < 1.0:
        return img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    return img


def show_slide_layers(fpath, thumb_w=800, thumb_h=600, max_native=4_000_000):
    """Display all pyramid levels + associated images of a slide as a panel row."""
    fname = os.path.basename(fpath)
    try:
        slide = openslide.OpenSlide(fpath)
    except Exception as exc:
        print(f"  \u26a0  Cannot open {fname}: {exc}")
        return
    layers = []
    mw, mh = slide.dimensions
    layers.append({
        "title": f"main\n{mw:,} \u00d7 {mh:,} px",
        "img":   slide.get_thumbnail((thumb_w, thumb_h)),
        "orig":  (mw, mh),
    })
    for lvl in range(1, slide.level_count):
        lw, lh = slide.level_dimensions[lvl]
        ds = slide.level_downsamples[lvl]
        if lw * lh <= max_native:
            img = slide.read_region((0, 0), lvl, (lw, lh))
        else:
            scale = min(thumb_w / lw, thumb_h / lh)
            img = slide.get_thumbnail((int(lw * scale), int(lh * scale)))
        layers.append({
            "title": f"level {lvl}  (\u00d7{ds:.0f}\u2193)\n{lw:,} \u00d7 {lh:,} px",
            "img":   img,
            "orig":  (lw, lh),
        })
    for name in sorted(slide.associated_images.keys()):
        img = slide.associated_images[name]
        aw, ah = img.size
        layers.append({"title": f"{name}\n{aw} \u00d7 {ah} px", "img": img, "orig": (aw, ah)})
    slide.close()
    n = len(layers)
    fig, axes = plt.subplots(1, n, figsize=(min(5 * n, 24), 4.5),
                             gridspec_kw={"wspace": 0.06})
    if n == 1:
        axes = [axes]
    for ax, layer in zip(axes, layers):
        disp = to_rgb(fit_to(layer["img"], thumb_w, thumb_h))
        dw, dh = disp.size
        ax.imshow(disp)
        ax.set_title(layer["title"], fontsize=8, fontweight="bold", pad=3, linespacing=1.4)
        ax.set_xlabel(f"displayed {dw}\u00d7{dh}", fontsize=7, labelpad=2)
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
        for spine in ax.spines.values():
            spine.set_linewidth(0.5)
    fig.suptitle(fname, fontsize=10, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()


def print_slide_metadata(fpath):
    """Print pyramid levels, associated images, and all openslide properties."""
    slide = openslide.OpenSlide(fpath)
    print(f"\n  Pyramid levels : {slide.level_count}")
    for lvl in range(slide.level_count):
        lw, lh = slide.level_dimensions[lvl]
        print(f"    [{lvl}] {lw:,} \u00d7 {lh:,} px  "
              f"(downsample \u00d7{slide.level_downsamples[lvl]:.2f})")
    assoc = sorted(slide.associated_images.keys())
    print(f"\n  Associated images : {assoc if assoc else '\u2014'}")
    print(f"\n  Properties ({len(slide.properties)}):")
    for k, v in sorted(slide.properties.items()):
        print(f"    {k} = {v}")
    slide.close()


# Discover on load so tiff_files is available to all downstream cells
tiff_files = discover_tiff_files()
print(f"Utilities loaded. {len(tiff_files)} {EXT} file(s) under {SOURCE}")

In [0]:
# Uses show_slide_layers + print_slide_metadata from the Shared utilities cell.
# Shows original source TIFFs only; synthetic variants are shown in cell 8.
for fpath in discover_tiff_files(synthetic=False):
    print(f"{'─' * 72}\n{os.path.basename(fpath)}")
    show_slide_layers(fpath)
    print_slide_metadata(fpath)
    print()


In [0]:
# 3rd TIFF file — display each layer large and save JPEGs back to the same volume.
import os
import openslide
from PIL import Image
import matplotlib.pyplot as plt

TARGET_FILE_IDX = 2       # Philips07
MAX_NATIVE_DIM  = 10_000  # read full level if max(w,h) ≤ this; else thumbnail
JPEG_THUMB_W    = 4096    # max width when level must be thumbnailed
JPEG_QUALITY    = 90


fpath = tiff_files[TARGET_FILE_IDX]
fname = os.path.basename(fpath)
fdir  = os.path.dirname(fpath)
stem  = os.path.splitext(fname)[0]
print(f"Source : {fpath}")
print(f"Output : {fdir}")
print(f"Stem   : {stem}\n")

slide = openslide.OpenSlide(fpath)

# ── collect layers ────────────────────────────────────────────────────────────
layers = {}

# main: high-res thumbnail from full pyramid
mw, mh = slide.dimensions
scale_main = JPEG_THUMB_W / max(mw, mh)
layers["main"] = {
    "img"  : slide.get_thumbnail((int(mw * scale_main), int(mh * scale_main))),
    "orig" : (mw, mh),
    "label": f"main  (level 0)\nfull: {mw:,} × {mh:,} px",
}

# pyramid levels 1 – N
for lvl in range(1, slide.level_count):
    lw, lh = slide.level_dimensions[lvl]
    ds = slide.level_downsamples[lvl]
    key = f"level_{lvl}"
    if max(lw, lh) <= MAX_NATIVE_DIM:          # safe to read entirely
        img = slide.read_region((0, 0), lvl, (lw, lh))
    else:                                       # too large — thumbnail
        sc = JPEG_THUMB_W / max(lw, lh)
        img = slide.get_thumbnail((int(lw * sc), int(lh * sc)))
    layers[key] = {
        "img"  : img,
        "orig" : (lw, lh),
        "label": f"level {lvl}  (×{ds:.0f}↓)\n{lw:,} × {lh:,} px",
    }

# associated sub-images (label / macro / thumbnail, if present)
for name in sorted(slide.associated_images.keys()):
    img = slide.associated_images[name]
    aw, ah = img.size
    layers[f"assoc_{name}"] = {
        "img"  : img,
        "orig" : (aw, ah),
        "label": f"{name}\n{aw} × {ah} px",
    }

slide.close()

# ── display: one large figure per layer ──────────────────────────────────────
for key, layer in layers.items():
    rgb = to_rgb(layer["img"])
    w, h = rgb.size
    fig_w = 14
    fig_h = fig_w * h / w
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.imshow(rgb)
    ax.set_title(
        f"{fname}  —  {layer['label']}",
        fontsize=11, fontweight="bold", pad=6, linespacing=1.5,
    )
    ax.set_xlabel(
        f"extracted at {w:,} × {h:,} px  |  original: {layer['orig'][0]:,} × {layer['orig'][1]:,} px",
        fontsize=9,
    )
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    plt.tight_layout()
    plt.show()

# ── save JPEGs ────────────────────────────────────────────────────────────────
print(f"\nSaving JPEGs to {fdir}\n")
for key, layer in layers.items():
    rgb      = to_rgb(layer["img"])
    jpg_name = f"{stem}__{key}.jpg"      # e.g. Philips07_3b946eed-...__level_3.jpg
    jpg_path = os.path.join(fdir, jpg_name)
    rgb.save(jpg_path, "JPEG", quality=JPEG_QUALITY)
    ow, oh = layer["orig"]
    jw, jh = rgb.size
    print(f"  {jpg_name}")
    print(f"    orig {ow:,}×{oh:,}  →  saved {jw:,}×{jh:,}  ({jpg_path})")

print(f"\n{len(layers)} JPEG(s) written.")


In [0]:
import os
import openslide
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fpath = tiff_files[2]   # Philips07
fname = os.path.basename(fpath)
slide = openslide.OpenSlide(fpath)

# ── blood vessel location estimated from level-6 visual (1,440 × 800 px) ─────
# Bright pink/magenta spot, lower-centre of tissue. Adjust if needed.
VESSEL_X_L6 = 620     # centre x in level-6 pixels
VESSEL_Y_L6 = 540     # centre y in level-6 pixels
CROP_R_L6   = 160     # half-side of crop square in level-6 pixels

DS = {lvl: int(slide.level_downsamples[lvl]) for lvl in range(slide.level_count)}

# ── crop box in level-6 coordinates ─────────────────────────────────────────
l6_x0 = max(0, VESSEL_X_L6 - CROP_R_L6)
l6_y0 = max(0, VESSEL_Y_L6 - CROP_R_L6)
l6_w  = min(CROP_R_L6 * 2, slide.level_dimensions[6][0] - l6_x0)
l6_h  = min(CROP_R_L6 * 2, slide.level_dimensions[6][1] - l6_y0)

# level-0 origin used by read_region for every pyramid level
loc0 = (l6_x0 * DS[6], l6_y0 * DS[6])

# ── read four views of the same region ─────────────────────────────────────────
lvl6_full = slide.read_region((0, 0), 6, slide.level_dimensions[6]).convert("RGB")
lvl6_crop = slide.read_region(loc0, 6, (l6_w,  l6_h)).convert("RGB")

# level 4  (×16 downsample) — 4× more detail than level 6
l4_w = l6_w * DS[6] // DS[4]
l4_h = l6_h * DS[6] // DS[4]
lvl4_crop = slide.read_region(loc0, 4, (l4_w, l4_h)).convert("RGB")

# level 2  (×4 downsample) — 16× more detail than level 6
l2_w = l6_w * DS[6] // DS[2]
l2_h = l6_h * DS[6] // DS[2]
lvl2_crop = slide.read_region(loc0, 2, (l2_w, l2_h)).convert("RGB")

slide.close()

# ── display ──────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(22, 7))
gs  = fig.add_gridspec(1, 4, wspace=0.05)
axes = [fig.add_subplot(gs[i]) for i in range(4)]

# Panel 1: level-6 overview with yellow crop-box annotation
axes[0].imshow(lvl6_full)
axes[0].add_patch(patches.Rectangle(
    (l6_x0, l6_y0), l6_w, l6_h,
    linewidth=2.5, edgecolor="yellow", facecolor="none",
))
axes[0].set_title("Level 6 — overview\n1,440 × 800 px  (×64↓)",
                  fontsize=9, fontweight="bold", pad=4)
axes[0].set_xlabel("yellow = crop region", fontsize=7)

# Panels 2-4: progressive zoom
for ax, img, lvl, label in [
    (axes[1], lvl6_crop, 6, "Level 6 crop"),
    (axes[2], lvl4_crop, 4, "Level 4  —  ×4 zoom"),
    (axes[3], lvl2_crop, 2, "Level 2  —  ×16 zoom"),
]:
    w, h = img.size
    ax.imshow(img)
    ax.set_title(
        f"{label}\n{w:,} × {h:,} px  (×{DS[lvl]}↓)",
        fontsize=9, fontweight="bold", pad=4,
    )
    ax.set_xlabel(f"origin L0: ({loc0[0]:,}, {loc0[1]:,})", fontsize=7)

for ax in axes:
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

fig.suptitle(f"{fname}  —  blood vessel zoom",
             fontsize=11, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print(f"Level-0 origin : {loc0}")
print(f"Level 6 crop   : {l6_w} × {l6_h} px")
print(f"Level 4 crop   : {l4_w} × {l4_h} px  (×4 more detail)")
print(f"Level 2 crop   : {l2_w} × {l2_h} px  (×16 more detail)")


In [0]:
from PIL.TiffTags import TAGS  # to_rgb, discover_tiff_files, tiff_files from Shared utilities

# Tags that could carry PHI
PHI_CANDIDATES = {
    270: "ImageDescription",
    305: "Software",
    315: "Artist",
    316: "HostComputer",
    33432: "Copyright",
    37510: "UserComment",
    40092: "XPComment",
    40094: "XPKeywords",
    40095: "XPSubject",
}

for fpath in tiff_files:
    fname = os.path.basename(fpath)
    print(f"\n{'\u2550' * 72}")
    print(f"{fname}")
    print(f"{'\u2500' * 72}")

    try:
        img = Image.open(fpath)
        n_frames = getattr(img, "n_frames", 1)
        print(f"  IFDs (frames): {n_frames}")

        # ── iterate all IFDs, deduplicate by tag-code signature ─────────────
        sig_map = {}
        for fi in range(n_frames):
            try:
                img.seek(fi)
            except EOFError:
                break
            tags = dict(getattr(img, "tag_v2", {}))
            sig = tuple(sorted(tags.keys()))
            if sig not in sig_map:
                sig_map[sig] = {"frames": [], "sample": tags}
            sig_map[sig]["frames"].append(fi)

        # ── also expose SubIFDs (tag 330) from IFD 0 if present ─────────
        img.seek(0)
        subifd_tag = getattr(img, "tag_v2", {}).get(330)
        if subifd_tag:
            print(f"  SubIFDs (tag 330): {subifd_tag}")

        img.close()

        # ── print each unique tag group ──────────────────────────────
        for sig, info in sig_map.items():
            frames = info["frames"]
            if len(frames) == 1:
                f_label = f"IFD {frames[0]}"
            elif frames == list(range(frames[0], frames[-1] + 1)):
                f_label = f"IFD {frames[0]}\u2013{frames[-1]}  ({len(frames)} frames)"
            else:
                f_label = f"{len(frames)} IFDs (non-contiguous)"

            tags = info["sample"]
            print(f"\n  \u2500\u2500 {f_label}  ({len(sig)} tags) \u2500\u2500")

            for code in sorted(tags.keys()):
                name = TAGS.get(code, f"Unknown_{code}")
                val  = tags[code]

                if isinstance(val, bytes):
                    try:
                        val_str = val.decode("utf-8", errors="replace").strip()
                    except Exception:
                        val_str = f"<bytes len={len(val)}>"
                elif isinstance(val, (tuple, list)) and len(val) > 8:
                    val_str = f"{type(val).__name__}[{len(val)}]  {repr(val[:4])} \u2026"
                else:
                    val_str = repr(val)

                if len(val_str) > 300:
                    val_str = val_str[:300] + " \u2026"

                phi = "  \u26a0\ufe0f  PHI?" if code in PHI_CANDIDATES else ""
                print(f"    {code:6d}  {name:<40s}  {val_str}{phi}")

    except Exception as exc:
        print(f"  ERROR: {exc}")

print("\nDone.")


In [0]:
# Writes one *__synthetic_phi.tiff per source file, containing:
#  - Aperio-style ImageDescription (tag 270) with fake patient PHI
#  - Software / Artist / HostComputer tags
#  - Rendered label sub-image with visible PHI text + barcode
#  - Macro sub-image (tissue thumbnail)
# Pyramid is built from source levels 5-8 (max 2,880 x 1,600) for speed.
# NOTE: tifffile requires random-access seeks; UC volumes don't support them.
# Strategy: write to /tmp, then shutil.copy2 to the volume.

import random, shutil, tempfile
import tifffile
import numpy as np
from PIL import ImageDraw  # to_rgb, discover_tiff_files, tiff_files from Shared utilities

# ── synthetic patient records ────────────────────────────────────────────────
SYNTHETIC_PATIENTS = [
    {"Patient": "Smith, John A",      "DOB": "1965-03-22", "MRN": "87654321",
     "AccessionNumber": "ACC-2023-001", "Clinic": "Oncology",
     "Pathologist": "Dr. Jane Doe",    "User": "jdoe"},
    {"Patient": "Johnson, Mary B",    "DOB": "1978-11-15", "MRN": "12345678",
     "AccessionNumber": "ACC-2023-002", "Clinic": "Pathology",
     "Pathologist": "Dr. Robert Chen", "User": "rchen"},
    {"Patient": "Williams, David C",  "DOB": "1952-07-04", "MRN": "99887766",
     "AccessionNumber": "ACC-2023-003", "Clinic": "Surgical",
     "Pathologist": "Dr. Sarah Kim",   "User": "skim"},
]

COPY_LEVELS = [5, 6, 7, 8]   # 2,880×1,600 → 360×200 — fast to read


def make_aperio_description(phi: dict, stem: str) -> str:
    """Aperio pipe-delimited ImageDescription with embedded PHI (tag 270)."""
    header = "Aperio Image Library v12.1.0\r\n[0,0 92160x51200] (512x512) JPEG/YCC Q=80"
    fields = {"Filename": stem, "Date": "2023-04-15", "Time": "10:23:45", **phi}
    body = "|".join(f"{k} = {v}" for k, v in fields.items())
    return f"{header}||{body}"


def make_label_image(phi: dict, size=(387, 463)) -> np.ndarray:
    """Render a patient-sticker image with clearly legible PHI text."""
    img  = Image.new("RGB", size, (255, 255, 255))
    draw = ImageDraw.Draw(img)
    draw.rectangle([2, 2, size[0]-3, size[1]-3], outline=(0, 0, 0), width=2)
    lines = [
        f"Patient : {phi['Patient']}",
        f"DOB     : {phi['DOB']}",
        f"MRN     : {phi['MRN']}",
        f"Accn    : {phi['AccessionNumber']}",
        f"Clinic  : {phi['Clinic']}",
        f"Path    : {phi['Pathologist']}",
        f"Date    : 2023-04-15",
        f"User    : {phi['User']}",
        "",
        "** SYNTHETIC PHI — TEST ONLY **",
        "** NOT A REAL PATIENT **",
    ]
    y = 16
    for line in lines:
        draw.text((10, y), line, fill=(0, 0, 0))
        y += 34
    # Fake barcode strip
    random.seed(42)
    for x in range(10, size[0]-10, 3):
        h = random.randint(15, 50)
        draw.rectangle([x, size[1]-70, x+1, size[1]-70+h], fill=(0, 0, 0))
    return np.array(img)


# ── write one synthetic TIFF per source file ──────────────────────────────────────
for fi, fpath in enumerate(tiff_files):
    # skip files that are already synthetic
    if "__synthetic_phi" in fpath:
        continue

    phi   = SYNTHETIC_PATIENTS[fi % len(SYNTHETIC_PATIENTS)]
    stem  = os.path.splitext(os.path.basename(fpath))[0]
    out_vol = os.path.join(os.path.dirname(fpath), f"{stem}__synthetic_phi.tiff")
    tmp_out = os.path.join(tempfile.gettempdir(), f"{stem}__synthetic_phi.tiff")

    print(f"\n{'\u2500'*72}")
    print(f"  source  : {os.path.basename(fpath)}")
    print(f"  subject : {phi['Patient']}  MRN={phi['MRN']}")
    print(f"  output  : {out_vol}")

    slide   = openslide.OpenSlide(fpath)
    pyramid = []
    for lvl in COPY_LEVELS:
        if lvl >= slide.level_count:
            continue
        lw, lh = slide.level_dimensions[lvl]
        arr = np.array(slide.read_region((0, 0), lvl, (lw, lh)).convert("RGB"))
        pyramid.append((lvl, arr, lw, lh))
        print(f"    level {lvl}: {lw}\u00d7{lh}")
    slide.close()

    if not pyramid:
        print("  \u26a0  no levels — skipping")
        continue

    image_desc = make_aperio_description(phi, stem)
    label_arr  = make_label_image(phi)
    macro_sm   = np.array(
        Image.fromarray(pyramid[0][1]).resize((640, 356), Image.LANCZOS)
    )
    # metadata=None disables tifffile's auto-JSON shape override so our
    # description= and extratags= values are written to tag 270 unchanged.
    _write = dict(photometric="rgb", tile=(512, 512),
                  compression="deflate", compressionargs={"level": 6},
                  metadata=None)

    with tifffile.TiffWriter(tmp_out, bigtiff=True) as tif:

        # IFD 0 — main image (level 5 of source) + all PHI tags
        _, arr, _, _ = pyramid[0]
        tif.write(
            arr, **_write,
            subfiletype=0,
            description=image_desc,             # tag 270 — Aperio-style PHI
            software="Philips IntelliSite 3.0",  # tag 305
            extratags=[
                (315, 2, 0, phi["User"],      True),   # Artist  — PHI
                (316, 2, 0, "SCANNER-PHI-01", True),   # HostComputer
            ],
        )

        # IFDs 1–N — reduced-resolution pyramid
        for _, arr, _, _ in pyramid[1:]:
            tif.write(arr, **_write, subfiletype=1)

        # label sub-image — PHI visible in pixel content (VLM/OCR target)
        tif.write(
            label_arr,
            photometric="rgb", compression="deflate",
            compressionargs={"level": 6},
            metadata=None, subfiletype=1, description="label",
        )

        # macro sub-image — tissue overview
        tif.write(
            macro_sm,
            photometric="rgb", compression="deflate",
            compressionargs={"level": 6},
            metadata=None, subfiletype=1, description="macro",
        )

    shutil.copy2(tmp_out, out_vol)
    os.remove(tmp_out)

    fsize = os.path.getsize(out_vol) / 1024 / 1024
    print(f"  written {fsize:.1f} MB  (deflate-compressed)")
    print(f"  desc   : {image_desc[:120]} \u2026")

    # ── verify: re-open with openslide and confirm PHI is readable ───────────
    try:
        chk = openslide.OpenSlide(out_vol)
        desc  = chk.properties.get("openslide.comment",
                chk.properties.get("tiff.ImageDescription", "(none)"))
        assoc = sorted(chk.associated_images.keys())
        print(f"  verify : vendor={chk.properties.get('openslide.vendor')}  "
              f"assoc={assoc}  desc_len={len(desc)}")
        print(f"  PHI ok : Patient={('Patient' in desc)}  "
              f"MRN={('MRN' in desc)}  User={('User' in desc)}")
        chk.close()
    except Exception as exc:
        print(f"  verify error: {exc}")

print("\nSynthetic PHI TIFFs complete.")


In [0]:
# Displays each synthetic PHI TIFF in full:
#   1. Pyramid layers panel (via show_slide_layers)
#   2. Full ImageDescription (tag 270) with PHI fields flagged ⚠️
#   3. label + macro sub-images read from PIL IFDs — the VLM / OCR target
# All shared helpers come from the Shared utilities cell.
from PIL.TiffTags import TAGS

PHI_KEYS = [
    "Patient", "DOB", "MRN", "AccessionNumber", "Clinic",
    "Pathologist", "Date", "Time", "User", "Filename", "ImageID",
]

synthetic_files = [f for f in discover_tiff_files() if "__synthetic_phi" in f]
print(f"Found {len(synthetic_files)} synthetic PHI TIFF(s)\n")

for fpath in synthetic_files:
    fname = os.path.basename(fpath)
    print(f"\n{'\u2550' * 72}\n{fname}\n{'\u2500' * 72}")

    # ── 1. Pyramid layers (openslide) ─────────────────────────────────────────
    show_slide_layers(fpath)

    # ── 2. ImageDescription via PIL tag_v2 (full string, not truncated) ────────
    pil_img = Image.open(fpath)
    pil_img.seek(0)
    raw = dict(getattr(pil_img, "tag_v2", {})).get(270, b"")
    desc = raw.decode("utf-8", errors="replace").strip("\x00") if isinstance(raw, bytes) else str(raw)

    print(f"\n  ImageDescription ({len(desc)} chars):")
    for part in desc.replace("\r\n", "||").split("|"):
        part = part.strip()
        if not part:
            continue
        is_phi = any(k in part for k in PHI_KEYS)
        marker = "  \u26a0\ufe0f PHI" if is_phi else ""
        print(f"    {part}{marker}")

    # ── 3. label / macro sub-images (PIL IFD walk) ─────────────────────────
    n_frames = getattr(pil_img, "n_frames", 1)
    sub_imgs  = []
    for fi in range(n_frames):
        try:
            pil_img.seek(fi)
        except EOFError:
            break
        ifd_desc = dict(getattr(pil_img, "tag_v2", {})).get(270, b"")
        if isinstance(ifd_desc, bytes):
            ifd_desc = ifd_desc.decode("utf-8", errors="replace").strip("\x00")
        if ifd_desc in ("label", "macro"):
            sub_imgs.append({"name": ifd_desc, "img": pil_img.copy()})
    pil_img.close()

    if sub_imgs:
        n = len(sub_imgs)
        fig, axes = plt.subplots(1, n, figsize=(9 * n, 9))
        if n == 1:
            axes = [axes]
        for ax, si in zip(axes, sub_imgs):
            rgb = to_rgb(si["img"])
            w, h = rgb.size
            ax.imshow(rgb)
            ax.set_title(
                f"{si['name']}  ({w} \u00d7 {h} px)",
                fontsize=12, fontweight="bold", pad=6,
            )
            ax.set_xlabel(
                "PHI rendered in pixels — VLM / OCR redaction target",
                fontsize=9,
            )
            ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
        fig.suptitle(
            f"{fname} — sub-images",
            fontsize=11, fontweight="bold", y=1.01,
        )
        plt.tight_layout()
        plt.show()
    else:
        print("\n  (no label/macro found via PIL IFD walk)")
    print()

print("Done.")


In [0]:
# Side-by-side structural and metadata comparison between:
#   - Philips BIG.tiff (OSUWMC) — metadata-bare
#   - Aperio SVS (orthanc_demo) — rich PHI metadata + associated sub-images
# Also shows the sub-images from SVS that are absent in the Philips files.

import glob, os
import openslide
from PIL import Image
import matplotlib.pyplot as plt
from PIL.TiffTags import TAGS

# to_rgb, discover_tiff_files, tiff_files from Shared utilities
orig_tiffs = discover_tiff_files(synthetic=False)

# Locate SVS files
svs_files = sorted(
    glob.glob("/Volumes/hls_pathology/orthanc_demo/raw_images/Aperio/**/*.svs",
              recursive=True) +
    glob.glob("/Volumes/hls_pathology/orthanc_demo/raw_images/Aperio/*.svs")
)[:3]  # first 3 for comparison

print(f"Philips TIFFs : {len(orig_tiffs)}")
print(f"Aperio SVS    : {len(svs_files)}")


def slide_profile(fpath):
    """Return a dict of key attributes for one openslide-readable file."""
    fname = os.path.basename(fpath)
    try:
        s = openslide.OpenSlide(fpath)
        desc  = s.properties.get("openslide.comment",
               s.properties.get("tiff.ImageDescription", ""))
        phi_keys = [k for k in
                    ["Date","Time","User","Filename",
                     "Patient","DOB","MRN","AccessionNumber",
                     "Clinic","Pathologist","Procedure","Diagnosis"]
                    if k in desc]
        profile = {
            "fname"       : fname,
            "vendor"      : s.properties.get("openslide.vendor", "?"),
            "dims"        : s.dimensions,
            "levels"      : s.level_count,
            "mpp"         : s.properties.get("openslide.mpp-x",
                            s.properties.get("aperio.MPP", "n/a")),
            "n_props"     : len(s.properties),
            "desc_len"    : len(desc),
            "phi_keys"    : phi_keys,
            "associated"  : sorted(s.associated_images.keys()),
            "assoc_images": {k: s.associated_images[k]
                             for k in s.associated_images.keys()},
            "properties"  : dict(s.properties),
            "slide"       : s,
        }
        return profile
    except Exception as exc:
        return {"fname": fname, "error": str(exc)}


print("\nProfiling files …")
svs_profiles  = [slide_profile(f) for f in svs_files]
tiff_profiles = [slide_profile(f) for f in orig_tiffs]


# ── text comparison table ────────────────────────────────────────────────────────────
vs  = svs_profiles[0]  if svs_profiles  else {}
vt  = tiff_profiles[0] if tiff_profiles else {}

print(f"\n{'\u2550'*90}")
print(f"{'Attribute':<28} {'Aperio SVS':<30} {'Philips BIG.tiff':<30}")
print(f"{'\u2500'*90}")
rows = [
    ("File",           lambda p: p.get("fname","?")[:40]),
    ("openslide.vendor",lambda p: p.get("vendor","?")),
    ("Full resolution", lambda p: f"{p['dims'][0]:,}\u00d7{p['dims'][1]:,}" if "dims" in p else "?"),
    ("Pyramid levels",  lambda p: str(p.get("levels","?"))),
    ("MPP (microns/px)",lambda p: str(p.get("mpp","?"))),
    ("# openslide props",lambda p: str(p.get("n_props","?"))),
    ("ImageDescription len",lambda p: str(p.get("desc_len",0)) + " chars"),
    ("PHI keys in desc", lambda p: str(p.get("phi_keys",[]))),
    ("Associated images",lambda p: str(p.get("associated",[]))),
]
for name, fn in rows:
    sv = fn(vs) if vs else "N/A"
    tv = fn(vt) if vt else "N/A"
    flag = " \u26a0\ufe0f" if name == "PHI keys in desc" and sv and sv != "[]" else ""
    print(f"  {name:<26} {sv:<30} {tv:<30}{flag}")
print(f"{'\u2550'*90}")


# ── print SVS ImageDescription (show PHI fields) ──────────────────────────────────
if vs and vs.get("desc_len", 0) > 0:
    print(f"\nSVS ImageDescription ({vs['fname']})")
    desc = vs["properties"].get("openslide.comment",
           vs["properties"].get("tiff.ImageDescription",""))
    for part in desc.split("|"):
        part = part.strip()
        if not part:
            continue
        is_phi = any(k in part for k in
                     ["Date","Time","User","Patient","DOB","MRN",
                      "Accession","Clinic","Pathologist","Filename","ImageID"])
        marker = "  \u26a0\ufe0f PHI" if is_phi else ""
        print(f"    {part}{marker}")


# ── display SVS associated sub-images (absent in Philips) ────────────────────────
for prof in svs_profiles:
    if "error" in prof or not prof.get("associated"):
        continue
    assoc = prof["assoc_images"]
    n = len(assoc)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, (name, img) in zip(axes, assoc.items()):
        rgb = to_rgb(img)
        ax.imshow(rgb)
        ax.set_title(
            f"{name}\n{rgb.size[0]}\u00d7{rgb.size[1]} px",
            fontsize=9, fontweight="bold", pad=3,
        )
        ax.tick_params(left=False, bottom=False,
                       labelleft=False, labelbottom=False)
    fig.suptitle(
        f"SVS associated images — {prof['fname']}\n"
        f"(PHI risk: label barcode + printed text; absent in Philips TIFFs)",
        fontsize=9, fontweight="bold",
    )
    plt.tight_layout()
    plt.show()

# close openslide handles
for p in svs_profiles + tiff_profiles:
    if "slide" in p:
        try:
            p["slide"].close()
        except Exception:
            pass

print("\nComparison complete.")
